In [0]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from nyc_taxi.config import PipelineConfig, ref_month_str
from nyc_taxi.bronze import build_bronze, write_partition
from nyc_taxi.spark_utils import configure_session

cfg = PipelineConfig()
configure_session(spark)

In [0]:
for taxi_type in cfg.taxi_types:
    table = cfg.table("bronze", f"{taxi_type}_tripdata")
    for m in cfg.months():
        df = build_bronze(spark, cfg.landing_path(taxi_type, m), taxi_type, m)
        write_partition(df, table, m)
        print(f"{table} · {ref_month_str(m)} gravado")

In [0]:
display(spark.sql(f"""
  SELECT 'yellow' AS taxi_type, ref_month, COUNT(*) AS rows FROM {cfg.table('bronze', 'yellow_tripdata')} GROUP BY ref_month
  UNION ALL
  SELECT 'green', ref_month, COUNT(*) FROM {cfg.table('bronze', 'green_tripdata')} GROUP BY ref_month
  ORDER BY taxi_type, ref_month"""))